# PySpark interview coding lab

Install a compatible PySpark distribution and the Java runtime required by that release; configure JAVA_HOME if needed. This notebook uses classic local Spark, not Spark Connect. It does not configure a production cluster.

Use only the synthetic fixtures supplied here. No private data is included. This notebook never installs packages automatically. Replace each `...`, run the exercise cell, then its separate test cell. Failures before completion are intentional. Tests cover selected cases, not every production input. Reference solutions are text at the end and are not executed automatically.


In [ ]:
from pyspark.sql import SparkSession
spark = (SparkSession.builder.master('local[2]')
         .appName('field-guide-exercises')
         .config('spark.sql.shuffle.partitions', '3')
         .getOrCreate())
print('Spark:', spark.version)


## 01 / DATAFRAME BASICS: Filter and aggregate with native expressions

Keep positive, non-null amounts and total them by user without collecting raw rows to Python.

### Hints
- **Keep positive amounts:** SQL comparison with null is not true, so filtering amount > 0 also excludes nulls.
- **Sum amounts with a stable output name:** Use a native aggregate rather than a Python loop over collected rows.

### Expected behavior
For u1 amounts [10, -2, null, 5] and u2 amount [7], output totals are u1=15 and u2=7. Row ordering is unspecified.


In [ ]:
from pyspark.sql import functions as F

def user_totals(events):
    valid = events.filter(...)
    return valid.groupBy("user_id").agg(...)


In [ ]:
events = spark.createDataFrame([('u1', 10), ('u1', -2), ('u1', None), ('u1', 5), ('u2', 7)], 'user_id string, amount long')
result = {row.user_id: row.total for row in user_totals(events).collect()}
assert result == {'u1': 15, 'u2': 7}
assert set(user_totals(events).columns) == {'user_id', 'total'}
print('PASS: native filter, null handling, aggregation, output schema')


## 02 / JOIN CORRECTNESS: Preserve the left-side grain

Join a small unique user dimension to events, preserve unmatched events, and keep the intended row count.

### Hints
- **Hint that this tiny test dimension can be broadcast:** The fixture is small. A production broadcast decision needs a measured size and memory budget.
- **Preserve every event, including unmatched users:** An inner join would discard events with no matching dimension record.

### Expected behavior
Three input events remain three output rows. u3 has a null segment because it does not exist in the unique user dimension.


In [ ]:
from pyspark.sql import functions as F

def attach_segment(events, users):
    # Contract: users contains at most one row per non-null user_id.
    return events.join(..., on="user_id", how=...)


In [ ]:
events = spark.createDataFrame([(1, 'u1'), (2, 'u2'), (3, 'u3')], 'event_id long, user_id string')
users = spark.createDataFrame([('u1', 'A'), ('u2', 'B')], 'user_id string, segment string')
rows = attach_segment(events, users).orderBy('event_id').collect()
assert len(rows) == 3
assert [row.segment for row in rows] == ['A', 'B', None]
duplicates = users.unionByName(users.filter(F.col('user_id') == 'u1'))
assert attach_segment(events, duplicates).count() == 4
print('PASS: left preservation, unmatched nulls, duplicate-key failure demonstration')


## 03 / WINDOW FUNCTIONS: Choose the latest record deterministically

Keep one event per user with an explicit tie-breaker, not an arbitrary dropDuplicates selection.

### Hints
- **Order newest timestamp first, then largest unique event ID:** Timestamp ties need a stable secondary key. The fixture uses integer event times and unique event IDs.
- **Number rows within the window:** row_number assigns exactly one row number 1 within each nonempty group.

### Expected behavior
u1 keeps event 3 rather than event 2 when timestamps tie; u2 keeps its only event.


In [ ]:
from pyspark.sql import Window
from pyspark.sql import functions as F

def latest_per_user(events):
    window = Window.partitionBy("user_id").orderBy(...)
    ranked = events.withColumn("rn", ...)
    return ranked.filter(F.col("rn") == 1).drop("rn")


In [ ]:
events = spark.createDataFrame([('u1', 1, 10), ('u1', 2, 20), ('u1', 3, 20), ('u2', 4, 5)], 'user_id string, event_id long, event_time long')
result = {row.user_id: row.event_id for row in latest_per_user(events).collect()}
assert result == {'u1': 3, 'u2': 4}
assert 'rn' not in latest_per_user(events).columns
print('PASS: per-user window, deterministic tie break, helper-column cleanup')


## 04 / EXECUTION & DATA PARITY: Partition without changing the data

Redistribute a bounded fixture by key while preserving every row and duplicate.

### Hints
- **Redistribute into the requested count by user_id:** repartition can redistribute by a key and usually incurs a shuffle. coalesce is not a general keyed redistribution substitute.

### Expected behavior
The result has the requested three partitions for this explicit repartition operation, and its multiset of rows equals the input, including duplicates.


In [ ]:
def partition_by_user(events, partitions: int):
    if partitions <= 0:
        raise ValueError("partitions must be positive")
    return ...


In [ ]:
events = spark.createDataFrame([('u1', 1), ('u1', 1), ('u2', 2)], 'user_id string, value long')
result = partition_by_user(events, 3)
assert result.rdd.getNumPartitions() == 3
assert events.exceptAll(result).count() == 0
assert result.exceptAll(events).count() == 0
try:
    partition_by_user(events, 0)
except ValueError:
    pass
else:
    raise AssertionError('Invalid partition count must be rejected')
print('PASS: partition count, bidirectional multiset parity, invalid input')


In [ ]:
spark.stop()


# Reference solutions

Try the exercises first. Equivalent implementations can be valid even when the browser's limited reference matcher does not accept them.


## Filter and aggregate with native expressions

```python
from pyspark.sql import functions as F

def user_totals(events):
    valid = events.filter(F.col("amount") > 0)
    return valid.groupBy("user_id").agg(F.sum("amount").alias("total"))
```

DataFrame expressions remain in Spark's query plan. Filtering null/negative values follows an explicit business rule. The tiny test collects only a bounded two-row fixture; production results should not be collected wholesale.


## Preserve the left-side grain

```python
from pyspark.sql import functions as F

def attach_segment(events, users):
    # Contract: users contains at most one row per non-null user_id.
    return events.join(F.broadcast(users), on="user_id", how="left")
```

Broadcasting changes a possible physical execution strategy, not join semantics. Uniqueness is an explicit input contract; the separate test demonstrates row multiplication if it is violated rather than silently dropping duplicates.


## Choose the latest record deterministically

```python
from pyspark.sql import Window
from pyspark.sql import functions as F

def latest_per_user(events):
    window = Window.partitionBy("user_id").orderBy(F.col("event_time").desc(), F.col("event_id").desc())
    ranked = events.withColumn("rn", F.row_number().over(window))
    return ranked.filter(F.col("rn") == 1).drop("rn")
```

A deterministic order makes repeated runs reproducible for this input contract. rank or dense_rank can retain multiple tied records; dropDuplicates does not specify which record wins. Define a null-ordering policy when production timestamps can be null.


## Partition without changing the data

```python
def partition_by_user(events, partitions: int):
    if partitions <= 0:
        raise ValueError("partitions must be positive")
    return events.repartition(partitions, "user_id")
```

Partitioning is an execution choice, not a deduplication rule or an ordering guarantee. Equal row counts alone are insufficient to establish parity; compare multiset differences on controlled data. Production parity checks should avoid unnecessarily collecting large data.
